# Exercise 1: Blockchain Structure & Tamper Detection

**COMP842 Applied Blockchains and Cryptocurrencies**

- Name: Sokhour Lay
- Student ID: 25314544

## Approach

To make this exercise more unique, the blockchain is built around an academic theme rather than around generic financial transactions from the class tutorial. It is implemented as an academic records ledger, in which each block stores real academic records such as enrolments, grades, and issued credentials. The ledger uses two classes. The `Block` class stores each block's records, a SHA-256 hash, and a Merkle root over the records, while the `Blockchain` class creates the genesis block automatically, appends new blocks, validates the chain, and reports the first invalid block. The implementation is organised into six parts. The full source code is available in the GitHub repository linked on the title page.

## Library:

In [1]:
import hashlib

from datetime import datetime

### 1. SHA-256 hash helper

A helper that turns any text into a 64-character SHA-256 fingerprint. Every part below relies on it.

In [2]:
def sha256(text):
    """Return the SHA-256 hash of a string, as a 64-character hex string."""
    return hashlib.sha256(text.encode()).hexdigest()

# Quick test: same input -> same hash; one character change -> totally different
print(sha256("hello"))
print(sha256("hello!"))

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
ce06092fb948d9ffac7d1a376e404b26b7575bcc11ee05a4615fef4fec3a308b


### 2. Merkle root

Combines all of a block's transaction hashes into a single "Merkle root" hash. Changing any transaction changes this root.

In [12]:
def merkle_root(transactions):
    """Combine all transaction hashes into one 'Merkle root' hash."""
    if not transactions:
        return sha256("")                               # empty block -> hash of empty text

    layer = [sha256(tx) for tx in transactions]         # 1. hash every transaction (the leaves)

    while len(layer) > 1:                               # 2. repeat until one hash is left
        if len(layer) % 2 == 1:                         #    odd number? duplicate the last one
            layer.append(layer[-1])
        layer = [sha256(layer[i] + layer[i + 1])        # 3. hash each neighbouring pair
                 for i in range(0, len(layer), 2)]

    return layer[0]                                     # 4. the single hash left = the root

In [13]:
print(merkle_root(["tx1", "tx2", "tx3"]))
print(merkle_root(["tx1", "tx2", "tx3-CHANGED"]))

fbf8b59f1ad5a1723f350e130dd75701c2b5c11a44b5ffc4e6ed48b2e1c34d8f
95e098c67015c62f87926200538f2abebfc92b87f11b427f636425aa65d510a2


### 3. Block class

Stores the block's transactions, the previous block's hash, its Merkle root, and its own SHA-256 hash.

In [5]:
class Block:
    def __init__(self, index, records, previous_hash):
        self.index = index
        self.records = records
        self.timestamp = datetime.now().isoformat(timespec="seconds")  # real creation time
        self.previous_hash = previous_hash
        self.merkle_root = merkle_root(records)
        self.hash = self.compute_hash()

    def compute_hash(self):
        content = f"{self.index}{self.timestamp}{self.previous_hash}{self.merkle_root}"
        return sha256(content)

#### Test Cell

In [6]:
sample_records = [
    "ENROL: S2231 enrolled in English 101 (2026-S2)",
    "GRADE: S2231, History 210, Final Exam, 88/100",
]

b = Block(1, sample_records, "0")
print("Index:      ", b.index)
print("Records:    ", b.records)
print("Merkle root:", b.merkle_root)
print("Previous hash:", b.previous_hash)
print("Block hash: ", b.hash)

Index:       1
Records:     ['ENROL: S2231 enrolled in English 101 (2026-S2)', 'GRADE: S2231, History 210, Final Exam, 88/100']
Merkle root: 4c7a012cd2d666d87e297d1995603f50c23749a3c25823876698b545a105dd19
Previous hash: 0
Block hash:  b099f7ca48c42d7e1bd231d540a91a12b0abbca991a3d21deab0c7f1997d4c4c


### 4. Blockchain class

Creates the genesis block automatically, appends new blocks, validates the chain, and reports the first invalid block.

In [7]:
class Blockchain:
    def __init__(self):
        self.chain = [self._genesis()]            # start with the genesis block

    def _genesis(self):
        return Block(0, ["GENESIS: Academic Records Ledger"], "0")

    def add_block(self, records):
        previous = self.chain[-1]                 # the last block currently in the chain
        new_block = Block(len(self.chain), records, previous.hash)
        self.chain.append(new_block)

    def first_invalid_block(self):
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]
            if current.merkle_root != merkle_root(current.records):
                return i                          # a record was changed
            if current.hash != current.compute_hash():
                return i                          # block hash no longer matches its contents
            if current.previous_hash != previous.hash:
                return i                          # link to the previous block is broken
        return None                               # no problems found

    def is_valid(self):
        return self.first_invalid_block() is None

#### Test Cell

In [8]:
ledger = Blockchain()
ledger.add_block(["ENROL: S2231 enrolled in English 101 (2026-S2)"])
ledger.add_block(["GRADE: S2231, History 210, Final Exam, 88/100"])

print("Number of blocks:", len(ledger.chain))
print("Chain valid?    :", ledger.is_valid())
print("First invalid   :", ledger.first_invalid_block())

Number of blocks: 3
Chain valid?    : True
First invalid   : None


### 5. Build a chain of 10 blocks

Create ten blocks with distinct data and confirm the whole chain is valid (this is the output *before* tampering).

In [9]:
academic_records = [
    ["ENROL: S2231 enrolled in English 101 (2026-S2)"],           # block 1
    ["ENROL: S2231 enrolled in History 210 (2026-S2)"],           # block 2
    ["GRADE: S2231, Mathematics 150, Midterm, 76/100"],           # block 3
    ["GRADE: S2231, Science 120, Lab Report, 92/100"],            # block 4
    ["GRADE: S2231, Geography 205, Final Exam, 68/100"],          # block 5
    ["GRADE: S2231, Computer Science 101, Assignment 1, 88/100"], # block 6
    ["ENROL: S2231 enrolled in Economics 110 (2026-S2)"],         # block 7
    ["GRADE: S2231, Art 130, Portfolio, 95/100"],                 # block 8
    ["COMPLETE: S2231 finished Physics 140 with grade A"],        # block 9
    ["AWARD: Certificate of Progress issued to S2231 (2026)"],    # block 10
]

ledger = Blockchain()
for records in academic_records:
    ledger.add_block(records)

#### Test Cell

In [10]:
def print_chain(blockchain):
    for block in blockchain.chain:
        print(f"Block {block.index}  |  {block.timestamp}")
        print(f"   Records     : {block.records}")
        print(f"   Merkle root : {block.merkle_root}")
        print(f"   Prev hash   : {block.previous_hash}")
        print(f"   Hash        : {block.hash}")
        print()

print_chain(ledger)
print("Total blocks :", len(ledger.chain))
print("Chain valid? :", ledger.is_valid())
print("First invalid:", ledger.first_invalid_block())

Block 0  |  2026-09-07T16:35:33
   Records     : ['GENESIS: Academic Records Ledger']
   Merkle root : e4b3bbb455e6c5967559b0367c3830a8a2d9b82ad6b9e8b39569fa2cc03eddcd
   Prev hash   : 0
   Hash        : 70950bf15d8a44dd25befbd0d5dff45ab37b908309e03c374e1a3f9a687904f7

Block 1  |  2026-09-07T16:35:33
   Records     : ['ENROL: S2231 enrolled in English 101 (2026-S2)']
   Merkle root : c0e55f2bb1ac6baddf909d94b7a58f67609aaebd83e3c9f09d945a39579c6be1
   Prev hash   : 70950bf15d8a44dd25befbd0d5dff45ab37b908309e03c374e1a3f9a687904f7
   Hash        : 1562ca0286a2008347d47057313ad96f79f4edd3519554bde68c4e6507ed5d9c

Block 2  |  2026-09-07T16:35:33
   Records     : ['ENROL: S2231 enrolled in History 210 (2026-S2)']
   Merkle root : 6c338b0e58b376fc3c4b174e4bcbc1f142d5549af511cae17d20476cee0cfcce
   Prev hash   : 1562ca0286a2008347d47057313ad96f79f4edd3519554bde68c4e6507ed5d9c
   Hash        : a97feb4f41eeed7d0895c349f9a26553d87c00e349372ccca5727c25edcdde04

Block 3  |  2026-09-07T16:35:33
   R

### 6. Tamper detection — modify block 5

Change block 5's data without recomputing its hash, then validate again. The chain should now report block 5 as the first invalid block.

In [11]:
print("=== Before tampering ===")
print("Block 5 records:", ledger.chain[5].records)
print("Chain valid?   :", ledger.is_valid())
print()

# Realistic attack: the student raises their own grade from 68 to 95
ledger.chain[5].records = ["GRADE: S2231, Geography 205, Final Exam, 95/100  <-- FORGED"]

print("=== After tampering block 5 ===")
print("Block 5 records:", ledger.chain[5].records)
print("Chain valid?   :", ledger.is_valid())
print("First invalid  :", ledger.first_invalid_block())

=== Before tampering ===
Block 5 records: ['GRADE: S2231, Geography 205, Final Exam, 68/100']
Chain valid?   : True

=== After tampering block 5 ===
Block 5 records: ['GRADE: S2231, Geography 205, Final Exam, 95/100  <-- FORGED']
Chain valid?   : False
First invalid  : 5


### Refelction Questions:

#### 1. Explain why modifying the transaction data in the 5th block causes the subsequent blocks to become invalid.

When block 5 is tampered with, our validator function identifies block 5 as the first invalid block, because the record was changed and no longer matches the Merkle root and hash that were stored when the block was created. In this blockchain, every block stores its own hash together with the hash of the block before it. Because task 6 asked us to modify the data in block 5 without recalculating its hash or Merkle root, the following blocks remain valid: the hash and Merkle root stored in block 5 are still the original values, so block 5 is still correctly linked to block 6. However, if the attacker attempted to recalculate block 5's hash or Merkle root, block 6 would then become invalid, because it still stores the original hash of block 5, and this failure would continue through every following block. As a result, a single tampered record cannot be concealed without recomputing every block that comes after it, which is the property that makes a blockchain tamper-evident (Nakamoto, 2008).

#### 2. What information should a blockchain validation program report when blockchain corruption or tampering is detected? Explain why each piece of information is useful.

A validator should report several pieces of information so that any tampering can be identified and investigated. First, it should report whether the chain is valid or invalid, because this immediately tells the user whether any record in the ledger has been altered. Second, it should report which block is the first invalid block, because detecting and locating corrupted blocks is essential for maintaining the integrity of the ledger and for recovering the affected data (Zhang et al., 2018). Reporting the first invalid block is particularly important, because every block after a broken link also becomes invalid, so the first invalid block indicates the true point of the attack. Third, it should report the reason for the failure, such as whether a record was changed, a block header was changed, or a link to the previous block was broken, because this indicates the type of attack and helps to decide how to respond. In our implementation, the validator already reports whether the chain is valid and the index of the first invalid block, and it could be extended to also report which of the three checks failed. This kind of reliable integrity checking is what allows a blockchain to provide a tamper-proof record, which is especially important for sensitive data such as academic credentials (Rama Reddy et al., 2021).

#### 3. Suggest one improvement that would make your blockchain more resistant to tampering. Explain how your proposed improvement enhances the security or integrity of the blockchain.

Proof of Work is an effective improvement that would make my blockchain more resistant to tampering, and it is also the mechanism that will be implemented in Exercise 2. By implementing Proof of Work, a block is only accepted once a valid hash has been found that meets a difficulty target. Finding this hash requires repeatedly changing a nonce and rehashing until the result begins with a required number of leading zeros. Because this search is computationally expensive, an attacker who alters block 5 must redo the Proof of Work for block 5 and for every block after it, since each block depends on the previous one. This makes tampering far more costly, because the attacker would need enormous computing power to regenerate the whole chain rather than simply editing a stored value (Nakamoto, 2008). Importantly, this protection works even when the ledger runs on a single machine, as the computational cost alone makes modifying past blocks impractical even after an attacker has fully compromised the system (Austin & Di Troia, 2022). For an academic records ledger, this means that a student or insider could not quietly rewrite a stored grade without repeating the expensive mining for that block and all later blocks.